# VERA Master Pipeline
## Visual Evidence–Report Alignment for Hallucination Detection in Clinical AI

This single notebook runs the entire VERA pipeline end-to-end on Kaggle.

**Before running:**
1. Click **Add Input** (right sidebar) → search "Indiana University Chest X-rays" by raddar → add it.
2. Click **Add-ons** → **Secrets** → create two secrets:
   - `HF_DATASET_TOKEN` (read-only token for ReXGradient)
   - `HF_MODEL_TOKEN` (fine-grained token for CheXagent/LLaVA-Med)
3. Turn on **GPU** (right sidebar → Accelerator → T4 x2 or P100).
4. Hit **Run All** or run cell by cell.

---
## 0. Environment Setup
Clone the repo, install dependencies, and load your HuggingFace tokens.

In [ ]:
# === STEP 0A: Load HuggingFace Tokens from Kaggle Secrets ===
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

try:
    os.environ["HF_DATASET_TOKEN"] = user_secrets.get_secret("HF_DATASET_TOKEN")
    print("✅ HF_DATASET_TOKEN loaded from Kaggle Secrets.")
except:
    print("⚠️ HF_DATASET_TOKEN not found.")

try:
    os.environ["HF_MODEL_TOKEN"] = user_secrets.get_secret("HF_MODEL_TOKEN")
    print("✅ HF_MODEL_TOKEN loaded from Kaggle Secrets.")
except:
    print("⚠️ HF_MODEL_TOKEN not found.")

In [ ]:
# === STEP 0B: Clone Repo and Install Requirements ===
!git clone https://github.com/4-thkind/VERA.git
%cd VERA
!pip install -q -r requirements.txt
!pip install -q datasets torch torchvision transformers accelerate bitsandbytes
!pip install -q spacy scispacy sentence-transformers scikit-learn seaborn
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
print("\nAll dependencies installed.")

In [ ]:
# === STEP 0C: Verify Environment ===
import sys
from pathlib import Path
import torch

PROJECT_ROOT = Path('/kaggle/working/VERA')
sys.path.insert(0, str(PROJECT_ROOT))

from config import IU_DATA_DIR, OUTPUT_DIR, REX_SUBSET_SIZE, PROCESSED_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"IU Data Dir:  {IU_DATA_DIR}")
print(f"Output Dir:   {OUTPUT_DIR}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Phase 1: Data Preparation
Load Indiana U from Kaggle mount, stream ReXGradient from HuggingFace, merge and split 70/20/10.

In [ ]:
from config import IU_DATA_DIR, OUTPUT_DIR, REX_SUBSET_SIZE, HF_DATASET_ID, HF_DATASET_TOKEN
from src.data_utils import load_indiana_u, load_rexgradient, combine_datasets
import matplotlib.pyplot as plt
from PIL import Image

print("Loading Indiana University Dataset...")
iu_samples = load_indiana_u()
print(f"Loaded {len(iu_samples)} Indiana U samples.")

if iu_samples:
    sample = iu_samples[0]
    print(f"  Sample: {sample['image_id']} — {sample['report'][:150]}...")

In [ ]:
print(f"Loading {REX_SUBSET_SIZE} ReXGradient samples via streaming...")
rex_samples = load_rexgradient()
print(f"Loaded {len(rex_samples)} ReXGradient samples.")

if rex_samples:
    sample = rex_samples[0]
    print(f"  Sample: {sample['image_id']} — {sample['report'][:150]}...")

In [ ]:
print("Merging datasets and generating 70/20/10 splits...")
splits = combine_datasets(iu_samples, rex_samples)

print("\nSplit sizes:")
for split_name, split_data in splits.items():
    print(f"  {split_name.capitalize()}: {len(split_data)} samples")

In [ ]:
# Sanity check: display one image from each source
train_data = splits['train']
iu_example = next((s for s in train_data if s['source'] == 'indiana_u'), None)
rex_example = next((s for s in train_data if s['source'] == 'rexgradient'), None)

examples = []
if iu_example: examples.append(('Indiana U', iu_example))
if rex_example: examples.append(('ReXGradient', rex_example))

if examples:
    fig, axes = plt.subplots(1, len(examples), figsize=(12, 6))
    if len(examples) == 1: axes = [axes]
    for i, (source_name, sample) in enumerate(examples):
        img = Image.open(sample['image_path']).convert('L')
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"{source_name}\nID: {sample['image_id']}\n\n{sample['report'][:120]}...", fontsize=10, wrap=True)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

---
## Phase 2: Model Inference + Attention Extraction (GPU)
Load CheXagent, generate reports, and extract cross-attention maps.

In [ ]:
from config import (
    PROCESSED_DIR, ATTENTION_DIR, HF_MODEL_TOKEN,
    CHEXAGENT_MODEL_ID, LLAVA_MED_MODEL_ID,
    DEFAULT_MODEL, MAX_NEW_TOKENS, REPORT_PROMPT,
    PATCH_GRID_CHEXAGENT, PATCH_GRID_LLAVA,
    NUM_ATTENTION_LAYERS, FIGURES_DIR,
)
from src.data_utils import load_json, save_json, load_image
from src.attention_extractor import (
    AttentionExtractor, load_chexagent, load_llava_med, process_batch
)
import numpy as np

# === MODEL SELECTION ===
USE_MODEL = "chexagent"
MAX_IMAGES = 50  # Set to None for full dataset
USE_4BIT = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if USE_MODEL == "chexagent":
    MODEL_ID = CHEXAGENT_MODEL_ID
    PATCH_GRID = PATCH_GRID_CHEXAGENT
else:
    MODEL_ID = LLAVA_MED_MODEL_ID
    PATCH_GRID = PATCH_GRID_LLAVA

print(f"Model: {MODEL_ID}")
print(f"Patch grid: {PATCH_GRID}")
print(f"Device: {device}")

In [ ]:
# Load model
print(f"Loading {MODEL_ID}...")
if USE_MODEL == "chexagent":
    model, tokenizer, image_processor = load_chexagent(
        model_id=MODEL_ID, hf_token=HF_MODEL_TOKEN, device=device, load_in_4bit=USE_4BIT)
else:
    model, tokenizer, image_processor = load_llava_med(
        model_id=MODEL_ID, hf_token=HF_MODEL_TOKEN, device=device, load_in_4bit=USE_4BIT)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params / 1e9:.2f}B")

# Initialize attention extractor
extractor = AttentionExtractor(
    model=model, tokenizer=tokenizer, image_processor=image_processor,
    patch_grid=PATCH_GRID, num_layers_to_use=NUM_ATTENTION_LAYERS, device=device)
print("AttentionExtractor initialized.")

In [ ]:
# Single image sanity check
from scipy.ndimage import zoom

test_data = load_json(str(PROCESSED_DIR / 'test.json'))
sample = test_data[0]
test_image = Image.open(sample['image_path']).convert('RGB')

result = extractor.extract_attention(image=test_image, prompt=REPORT_PROMPT, max_new_tokens=MAX_NEW_TOKENS)

print(f"Generated report ({result['num_generated_tokens']} tokens):")
print(result['generated_text'])
print(f"\nAttention map shape: {result['attention_maps'].shape}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(test_image); axes[0].set_title('Original CXR'); axes[0].axis('off')
avg_attention = result['attention_maps'].mean(axis=0)
H, W = np.array(test_image).shape[:2]
attn_up = zoom(avg_attention, (H / avg_attention.shape[0], W / avg_attention.shape[1]), order=1)
axes[1].imshow(test_image); axes[1].imshow(attn_up, cmap='jet', alpha=0.5)
axes[1].set_title('Attention Heatmap'); axes[1].axis('off')
axes[2].imshow(avg_attention, cmap='hot', interpolation='nearest')
axes[2].set_title(f'Raw Attention Grid ({PATCH_GRID[0]}x{PATCH_GRID[1]})'); axes[2].axis('off')
plt.suptitle(f'Attention Sanity Check — {sample["image_id"]}', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(str(FIGURES_DIR / 'attention_sanity_check.png'), dpi=150); plt.show()

In [ ]:
# Batch inference
all_data = []
for split in ['train', 'val', 'test']:
    split_data = load_json(str(PROCESSED_DIR / f'{split}.json'))
    for entry in split_data: entry['split'] = split
    all_data.extend(split_data)

if MAX_IMAGES is not None:
    all_data = all_data[:MAX_IMAGES]

print(f"Running batch inference on {len(all_data)} images...")
output_dir = str(ATTENTION_DIR / USE_MODEL)
results = process_batch(extractor=extractor, data=all_data, output_dir=output_dir,
                        prompt=REPORT_PROMPT, max_new_tokens=MAX_NEW_TOKENS, save_every=10)

successful = [r for r in results if 'error' not in r]
failed = [r for r in results if 'error' in r]
print(f"\nProcessed: {len(successful)}/{len(results)}")
if failed:
    print(f"Failed: {len(failed)}")

# Save metadata
inference_meta = {
    'model_id': MODEL_ID, 'model_name': USE_MODEL, 'patch_grid': PATCH_GRID,
    'num_layers_used': NUM_ATTENTION_LAYERS, 'max_new_tokens': MAX_NEW_TOKENS,
    'num_processed': len(successful), 'num_failed': len(failed), 'prompt': REPORT_PROMPT,
}
save_json(inference_meta, str(ATTENTION_DIR / USE_MODEL / 'inference_meta.json'))

---
## Phase 3: Claim Extraction
Extract structured anatomical claims from generated reports using scispaCy.

In [ ]:
from config import CLAIMS_DIR
from src.claim_extractor import load_nlp_model, extract_claims, detect_relational_hallucinations, summarize_claims
import json
from tqdm import tqdm
from collections import Counter

nlp_model = load_nlp_model("en_core_sci_sm")
print(f"NLP model loaded: {nlp_model.meta['name']}" if nlp_model else "Using rule-based extraction only.")

# Load inference results
inference_results = load_json(str(ATTENTION_DIR / USE_MODEL / 'inference_results.json'))
inference_results = [r for r in inference_results if 'error' not in r]
print(f"Loaded {len(inference_results)} inference results")

In [ ]:
# Batch claim extraction
claims_output_dir = CLAIMS_DIR / USE_MODEL
claims_output_dir.mkdir(parents=True, exist_ok=True)

all_claims_summary = []
total_claims = 0
total_relational = 0

for entry in tqdm(inference_results, desc="Extracting claims"):
    report = entry.get('generated_report', '')
    if not report: continue
    claims = extract_claims(report, nlp_model, include_negated=True)
    relational_flags = detect_relational_hallucinations(report)
    claims_data = {
        'image_id': entry['image_id'], 'generated_report': report,
        'claims': claims, 'relational_flags': relational_flags,
        'num_claims': len(claims), 'num_relational_flags': len(relational_flags),
    }
    with open(claims_output_dir / f"{entry['image_id']}_claims.json", 'w') as f:
        json.dump(claims_data, f, indent=2)
    total_claims += len(claims)
    total_relational += len(relational_flags)
    all_claims_summary.append(claims_data)

print(f"\nTotal claims: {total_claims}")
print(f"Avg per report: {total_claims / len(inference_results):.1f}")
print(f"Relational flags: {total_relational}")

In [ ]:
# Claims statistics and visualization
all_claims = []
for entry in all_claims_summary:
    all_claims.extend(entry['claims'])

findings_counter = Counter(c['finding'] for c in all_claims)
top_findings = findings_counter.most_common(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
names, counts = zip(*top_findings) if top_findings else ([], [])
axes[0].barh(range(len(names)), counts, color='#3498db')
axes[0].set_yticks(range(len(names))); axes[0].set_yticklabels(names)
axes[0].set_xlabel('Count'); axes[0].set_title('Top 15 Findings', fontweight='bold'); axes[0].invert_yaxis()

locations_counter = Counter(c['location'] for c in all_claims if c['location'])
top_locations = locations_counter.most_common(10)
if top_locations:
    loc_names, loc_counts = zip(*top_locations)
    axes[1].barh(range(len(loc_names)), loc_counts, color='#2ecc71')
    axes[1].set_yticks(range(len(loc_names))); axes[1].set_yticklabels(loc_names)
    axes[1].set_xlabel('Count')
axes[1].set_title('Top 10 Anatomical Locations', fontweight='bold'); axes[1].invert_yaxis()
plt.suptitle('Claim Extraction Summary', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(str(FIGURES_DIR / 'claim_extraction_summary.png'), dpi=150); plt.show()

---
## Phase 4: Anatomy Atlas Validation
Validate the 13-zone chest anatomy atlas and generate patch masks.

In [ ]:
from config import CHEST_ZONES, LOCATION_MAPPING, PATCH_GRID_CHEXAGENT
from src.anatomy_atlas import (
    location_to_zones, claim_to_region, bbox_to_patch_mask,
    get_zone_colors, visualize_atlas_on_image, CHEST_ZONES as ATLAS_ZONES
)
import matplotlib.patches as mpatches

# Display all atlas zones
print("CHEST ANATOMY ATLAS — 13 Zones")
print("=" * 60)
for zone, bbox in ATLAS_ZONES.items():
    print(f"  {zone:<30} ({bbox[0]:.2f}, {bbox[1]:.2f}, {bbox[2]:.2f}, {bbox[3]:.2f})")

In [ ]:
# Visualize atlas on blank canvas
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
colors = get_zone_colors()
for zone_name, bbox in ATLAS_ZONES.items():
    x1, y1, x2, y2 = bbox
    color = np.array(colors[zone_name]) / 255.0
    rect = mpatches.FancyBboxPatch((x1, y1), x2-x1, y2-y1, boxstyle="round,pad=0.005",
        facecolor=(*color, 0.4), edgecolor=(*color, 1.0), linewidth=2)
    ax.add_patch(rect)
    ax.text(x1+(x2-x1)/2, y1+(y2-y1)/2, zone_name.replace('_', ' ').title(),
        ha='center', va='center', fontsize=7, fontweight='bold',
        color='black', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('VERA Chest Anatomy Atlas — 13 Zones', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(str(FIGURES_DIR / 'atlas_zones.png'), dpi=200); plt.show()

In [ ]:
# Overlay atlas on real CXR image
try:
    data = load_json(str(PROCESSED_DIR / 'test.json'))
    sample_image = np.array(Image.open(data[0]['image_path']).convert('RGB').resize((512, 512)))
except: sample_image = np.ones((512, 512, 3), dtype=np.uint8) * 200

overlaid = visualize_atlas_on_image(sample_image, alpha=0.35)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(sample_image); axes[0].set_title('Original CXR'); axes[0].axis('off')
axes[1].imshow(overlaid); axes[1].set_title('CXR with Atlas Overlay'); axes[1].axis('off')
plt.suptitle('Anatomy Atlas Validation', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(str(FIGURES_DIR / 'atlas_overlay.png'), dpi=200); plt.show()

In [ ]:
# Generate and visualize patch masks
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes_flat = axes.flatten()
zone_names = list(ATLAS_ZONES.keys())
for i, zn in enumerate(zone_names[:13]):
    mask = bbox_to_patch_mask(ATLAS_ZONES[zn], PATCH_GRID)
    axes_flat[i].imshow(mask, cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    axes_flat[i].set_title(zn.replace('_', '\n'), fontsize=9)
for i in range(13, 15): axes_flat[i].axis('off')
plt.suptitle(f'Patch Masks — {PATCH_GRID[0]}x{PATCH_GRID[1]} Grid', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(str(FIGURES_DIR / 'atlas_patch_masks.png'), dpi=150); plt.show()

# Export atlas
atlas_export = {
    'zones': {k: list(v) for k, v in ATLAS_ZONES.items()},
    'location_mapping': {k: v for k, v in LOCATION_MAPPING.items()},
    'patch_grid': list(PATCH_GRID), 'num_zones': len(ATLAS_ZONES),
}
with open(PROCESSED_DIR / 'anatomy_atlas.json', 'w') as f:
    json.dump(atlas_export, f, indent=2)
print(f"Atlas exported to: {PROCESSED_DIR / 'anatomy_atlas.json'}")

---
## Phase 5: VERA Scoring
Compute alignment scores, calibrate thresholds, and flag hallucinations.

In [ ]:
from config import RESULTS_DIR, SEVERITY_THRESHOLDS
from src.vera_scorer import (
    score_all_claims, calibrate_thresholds, SEVERITY_THRESHOLDS as DEFAULT_THRESHOLDS
)
from src.anatomy_atlas import claim_to_region
from collections import defaultdict

# Build split map
split_map = {}
for split in ['train', 'val', 'test']:
    for entry in load_json(str(PROCESSED_DIR / f'{split}.json')):
        split_map[entry['image_id']] = split
for r in inference_results:
    r['split'] = split_map.get(r['image_id'], 'unknown')

In [ ]:
# Score all claims
all_scored = []
all_claims_flat = []
claims_dir = CLAIMS_DIR / USE_MODEL
attention_dir = ATTENTION_DIR / USE_MODEL

for entry in tqdm(inference_results, desc="VERA scoring"):
    image_id = entry['image_id']
    claims_path = claims_dir / f"{image_id}_claims.json"
    attn_path = attention_dir / f"{image_id}_attention.npz"
    if not claims_path.exists() or not attn_path.exists(): continue

    with open(claims_path, 'r') as f: claims_data = json.load(f)
    claims = claims_data.get('claims', [])
    attention_maps = np.load(str(attn_path))['attention_maps']
    scored_claims = score_all_claims(claims, attention_maps, PATCH_GRID, DEFAULT_THRESHOLDS)

    result = {
        'image_id': image_id, 'split': entry.get('split', 'unknown'),
        'generated_report': entry.get('generated_report', ''),
        'claims': scored_claims, 'num_claims': len(scored_claims),
        'num_flagged': sum(1 for c in scored_claims if c.get('vera_flagged')),
    }
    all_scored.append(result)
    for c in scored_claims:
        c['image_id'] = image_id; c['split'] = entry.get('split', 'unknown')
    all_claims_flat.extend(scored_claims)

total_claims = len(all_claims_flat)
localizable = sum(1 for c in all_claims_flat if c.get('localizable'))
flagged = sum(1 for c in all_claims_flat if c.get('vera_flagged'))
print(f"\nTotal claims: {total_claims}")
print(f"Localizable: {localizable} ({localizable/total_claims*100:.1f}%)")
print(f"Flagged: {flagged} ({flagged/total_claims*100:.1f}%)")

In [ ]:
# VERA score distribution plot
import seaborn as sns

localizable_claims = [c for c in all_claims_flat if c.get('localizable')]
vera_scores = [c['vera_score'] for c in localizable_claims]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
if vera_scores:
    axes[0].hist(vera_scores, bins=30, color='#3498db', alpha=0.7, edgecolor='white')
    for tier, thresh in DEFAULT_THRESHOLDS.items():
        axes[0].axvline(x=thresh, linestyle='--', alpha=0.7, label=f'{tier}: T={thresh}')
    axes[0].set_xlabel('VERA Score'); axes[0].set_ylabel('Count')
    axes[0].set_title('VERA Score Distribution', fontweight='bold'); axes[0].legend(fontsize=9)

severity_data = defaultdict(list)
for c in localizable_claims:
    severity_data[c.get('severity_tier', 'unknown')].append(c['vera_score'])
tier_colors = {'critical': '#e74c3c', 'moderate': '#f39c12', 'mild': '#2ecc71'}
data_for_box = []; labels = []
for tier in ['critical', 'moderate', 'mild']:
    if tier in severity_data:
        data_for_box.append(severity_data[tier])
        labels.append(f"{tier}\n(n={len(severity_data[tier])})")
if data_for_box:
    bp = axes[1].boxplot(data_for_box, labels=labels, patch_artist=True)
    for patch, tier in zip(bp['boxes'], ['critical', 'moderate', 'mild']):
        patch.set_facecolor(tier_colors.get(tier, '#3498db')); patch.set_alpha(0.6)
    axes[1].set_ylabel('VERA Score'); axes[1].set_title('VERA Score by Severity Tier', fontweight='bold')
plt.tight_layout(); plt.savefig(str(FIGURES_DIR / 'vera_score_distribution.png'), dpi=200); plt.show()

In [ ]:
# Threshold calibration on validation split
val_data_ref = load_json(str(PROCESSED_DIR / 'val.json'))
ref_map = {entry['image_id']: entry.get('reference_report', '') for entry in val_data_ref}
val_claims = [c for c in all_claims_flat if c.get('split') == 'val' and c.get('localizable')]

val_gt = []
for claim in val_claims:
    ref = ref_map.get(claim.get('image_id', ''), '').lower()
    val_gt.append(claim.get('finding', '').lower() not in ref if ref else True)

val_score_dicts = [{'vera_score': c['vera_score'], 'severity_tier': c.get('severity_tier', 'moderate')} for c in val_claims]
calibrated_thresholds = calibrate_thresholds(val_score_dicts, val_gt, threshold_range=(0.05, 0.60), threshold_step=0.025)

print("Calibrated thresholds:")
for tier, t in calibrated_thresholds.items():
    print(f"  {tier}: {t:.3f} (default: {DEFAULT_THRESHOLDS.get(tier, 0.25)})")
save_json(calibrated_thresholds, str(RESULTS_DIR / 'calibrated_thresholds.json'))

# Re-score with calibrated thresholds
for claim in all_claims_flat:
    if not claim.get('localizable') or claim.get('vera_score') is None: continue
    tier = claim.get('severity_tier', 'moderate')
    claim['vera_threshold'] = calibrated_thresholds.get(tier, DEFAULT_THRESHOLDS.get(tier, 0.25))
    claim['vera_flagged'] = claim['vera_score'] < claim['vera_threshold']

new_flagged = sum(1 for c in all_claims_flat if c.get('vera_flagged'))
print(f"\nFlagged (calibrated): {new_flagged}/{total_claims} ({new_flagged/total_claims*100:.1f}%)")

# Save results
results_output = RESULTS_DIR / USE_MODEL
results_output.mkdir(parents=True, exist_ok=True)
save_json(all_scored, str(results_output / 'vera_scores.json'))
save_json(all_claims_flat, str(results_output / 'vera_claims_flat.json'))

---
## Phase 6: Final Evaluation
Generate ground truth labels, compute metrics, and produce all paper figures and tables.

In [ ]:
from config import NLI_MODEL_ID
from src.evaluation import (
    load_nli_model, compute_ground_truth_nli, compute_metrics,
    compute_per_severity_metrics, random_baseline,
    compute_roc, plot_vera_distribution, plot_roc_curve,
    plot_threshold_sensitivity, plot_comparison_figure,
    generate_results_table,
)
import pandas as pd

# Load NLI model and reference reports
print("Loading NLI model...")
nli_model = load_nli_model(NLI_MODEL_ID)

ref_reports = {}
for split in ['train', 'val', 'test']:
    for entry in load_json(str(PROCESSED_DIR / f'{split}.json')):
        ref_reports[entry['image_id']] = entry.get('reference_report', '')
print(f"Loaded {len(ref_reports)} reference reports")

In [ ]:
# Compute ground truth for test split
test_claims = [c for c in all_claims_flat if c.get('split') == 'test' and c.get('localizable')]
print(f"Computing ground truth for {len(test_claims)} test claims...")

claims_by_image = defaultdict(list)
for c in test_claims: claims_by_image[c['image_id']].append(c)

ground_truth = []
for image_id, img_claims in tqdm(claims_by_image.items(), desc="NLI ground truth"):
    ref = ref_reports.get(image_id, '')
    gt_labels = compute_ground_truth_nli(img_claims, ref, nli_model)
    ground_truth.extend(gt_labels)

print(f"Hallucination rate: {sum(ground_truth)/len(ground_truth)*100:.1f}%")

In [ ]:
# VERA metrics
vera_predictions = [c.get('vera_flagged', False) for c in test_claims]
vera_test_scores = [c.get('vera_score', 0.5) for c in test_claims]
vera_metrics = compute_metrics(vera_predictions, ground_truth)

print("=" * 60)
print("VERA RESULTS ON TEST SET")
print("=" * 60)
print(f"  Precision: {vera_metrics['precision']*100:.1f}%")
print(f"  Recall:    {vera_metrics['recall']*100:.1f}%")
print(f"  F1:        {vera_metrics['f1']*100:.1f}%")
print(f"  Accuracy:  {vera_metrics['accuracy']*100:.1f}%")

In [ ]:
# Baselines
hallucination_rate = sum(ground_truth) / len(ground_truth)
random_preds = random_baseline(len(test_claims), hallucination_rate, seed=42)
random_metrics = compute_metrics(random_preds, ground_truth)

test_entropies = [c.get('attention_entropy', 0) for c in test_claims]
if any(e > 0 for e in test_entropies):
    confidence_preds = [e > np.median(test_entropies) for e in test_entropies]
    confidence_metrics = compute_metrics(confidence_preds, ground_truth)
else:
    confidence_metrics = {'precision': 0.5, 'recall': 0.5, 'f1': 0.5}

# Table 1
method_results = {
    'Random Baseline': {**random_metrics, 'requires_labels': 'No'},
    'Confidence Threshold': {**confidence_metrics, 'requires_labels': 'No'},
    'NLI-Only': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'requires_labels': 'Yes'},
    'VERA (ours)': {**vera_metrics, 'requires_labels': 'No'},
}
table_md = generate_results_table(method_results, save_path=str(RESULTS_DIR / 'table1_main_results.csv'))
print("\nTABLE 1: MAIN RESULTS")
print(table_md)

In [ ]:
# Per-severity metrics
per_severity = compute_per_severity_metrics(test_claims, ground_truth)
print("\nPER-SEVERITY TIER RESULTS")
for tier, metrics in per_severity.items():
    print(f"  {tier.upper()}: P={metrics['precision']*100:.1f}%, R={metrics['recall']*100:.1f}%, F1={metrics['f1']*100:.1f}%")

In [ ]:
# Figure 2: VERA score distribution (hallucinated vs clean)
scores_hall = [c.get('vera_score') for c, gt in zip(test_claims, ground_truth) if gt and c.get('vera_score') is not None]
scores_clean = [c.get('vera_score') for c, gt in zip(test_claims, ground_truth) if not gt and c.get('vera_score') is not None]

print(f"Hallucinated claims: {len(scores_hall)}, mean VERA = {np.mean(scores_hall):.3f}")
print(f"Clean claims: {len(scores_clean)}, mean VERA = {np.mean(scores_clean):.3f}")

plot_vera_distribution(scores_hall, scores_clean,
    save_path=str(FIGURES_DIR / 'figure2_vera_distribution.png'),
    title='VERA Score Distribution: Hallucinated vs. Clean Claims')

In [ ]:
# Figure 3: Threshold sensitivity
test_severity_tiers = [c.get('severity_tier', 'moderate') for c in test_claims]
plot_threshold_sensitivity(vera_test_scores, ground_truth, test_severity_tiers,
    save_path=str(FIGURES_DIR / 'figure3_threshold_sensitivity.png'),
    title='Threshold Sensitivity: F1 vs. T per Severity Tier')

In [ ]:
# ROC Curve + AUROC
fpr, tpr, auroc = compute_roc(vera_test_scores, ground_truth)
print(f"AUROC: {auroc:.3f}")
plot_roc_curve(fpr=fpr, tpr=tpr, auroc=auroc,
    save_path=str(FIGURES_DIR / 'roc_curve.png'),
    title=f'VERA ROC Curve (AUROC = {auroc:.3f})')

In [ ]:
# Final summary
final_results = {
    'model': USE_MODEL, 'vera_metrics': vera_metrics,
    'random_baseline_metrics': random_metrics, 'confidence_baseline_metrics': confidence_metrics,
    'per_severity_metrics': {k: v for k, v in per_severity.items()},
    'auroc': float(auroc), 'hallucination_rate': float(sum(ground_truth) / len(ground_truth)),
    'num_test_claims': len(test_claims),
    'avg_vera_hallucinated': float(np.mean(scores_hall)) if scores_hall else 0,
    'avg_vera_clean': float(np.mean(scores_clean)) if scores_clean else 0,
}
save_json(final_results, str(RESULTS_DIR / 'final_results.json'))

print("\n" + "=" * 70)
print("VERA EVALUATION COMPLETE")
print("=" * 70)
print(f"Model: {USE_MODEL}")
print(f"Test claims: {len(test_claims)}")
print(f"Hallucination rate: {final_results['hallucination_rate']*100:.1f}%")
print(f"\nVERA Performance:")
print(f"  Precision: {vera_metrics['precision']*100:.1f}%")
print(f"  Recall:    {vera_metrics['recall']*100:.1f}%")
print(f"  F1:        {vera_metrics['f1']*100:.1f}%")
print(f"  AUROC:     {auroc:.3f}")
print(f"\nAll figures saved to: {FIGURES_DIR}")
print(f"All results saved to: {RESULTS_DIR}")

In [ ]:
# List all generated files
print("Generated Files:")
print("Figures:")
for f in sorted(FIGURES_DIR.glob('*.png')): print(f"  {f.name}")
print("\nResults:")
for f in sorted(RESULTS_DIR.rglob('*.json')): print(f"  {f.relative_to(RESULTS_DIR)}")
for f in sorted(RESULTS_DIR.rglob('*.csv')): print(f"  {f.relative_to(RESULTS_DIR)}")